In [ ]:
import random
import joblib
from nn_models   import *
from nn_data     import *
from nn_training import *
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_PATH    = '/path'
BEST_HP_PATH = '/path'

### 1. Load data

In [ ]:
ts_features       = TS_FEATURES
static_features   = STATIC_FEATURES
features          = ts_features + static_features
df_train = pd.read_csv(DATA_PATH + 'train.csv')
df_test  = pd.read_csv(DATA_PATH + 'test.csv')

Xy_train = df_train.copy()
Xy_test  = df_test.copy()

### 2. Standardize data

In [ ]:
# Fit scalers separately for ts and static features
ts_scaler     = StandardScaler()
static_scaler = StandardScaler()
target_scaler = StandardScaler()

ts_scaler.fit(np.array(Xy_train[ts_features]))
static_scaler.fit(np.array(Xy_train[static_features]))
target_scaler.fit(np.array(Xy_train['pm25_filled']).reshape(-1, 1))

# Apply transformation
Xy_train[ts_features]     = ts_scaler.transform(Xy_train[ts_features].values)
Xy_train[static_features] = static_scaler.transform(Xy_train[static_features].values)
Xy_train['pm25_filled']   = target_scaler.transform(Xy_train['pm25_filled'].values.reshape(-1, 1))
Xy_test[ts_features]      = ts_scaler.transform(Xy_test[ts_features].values)
Xy_test[static_features]  = static_scaler.transform(Xy_test[static_features].values)
Xy_test['pm25']           = transform_target_preserve_nan(Xy_test['pm25'], target_scaler) 

# Group by sequence length
train_groups = prepare_3d_input(df=Xy_train, ts_cols=ts_features, static_cols=static_features, 
                                target_col='pm25_filled', verbose=False)
test_groups  = prepare_3d_input(df=Xy_test,  ts_cols=ts_features, static_cols=static_features, verbose=False)

train_tensors = groups_to_tensors(train_groups, mode='db', reverse_sort=True)

# Quick check
for T, (X_ts, X_static, y_t) in train_tensors.items():
    print(f"Train group T={T:4d} | X_ts: {tuple(X_ts.shape)} | X_static: {tuple(X_static.shape)} | y: {tuple(y_t.shape)}")

### 3. Train the model

In [ ]:
loaded_study = joblib.load(BEST_HP_PATH)
best         = loaded_study.best_params
best_trial   = loaded_study.best_trial
best_epoch   = best_trial.user_attrs["best_epoch"]

In [ ]:
seed_everything()
model    = DB(n_fts       = N_FTS, 
              n_fstatic     = N_FSTATIC, 
              hts1          = best_trial.user_attrs['hts1'], 
              hts2          = best_trial.user_attrs['hts2'], 
              hts3          = best_trial.user_attrs['hts3'], 
              hstt1         = best_trial.user_attrs['hstt1'], 
              hstt2         = best_trial.user_attrs['hstt2'],
              drop_rate_ts  = best['drop_rate_ts'],
              drop_rate_stt = best['drop_rate_stt'])

optimizer      = torch.optim.Adam(model.parameters(), lr=best['lr'])
criterion      = nn.MSELoss()
epochs         = best_epoch
train_losses   = train_DB(model, train_tensors, criterion, optimizer, epochs) # X: [B, T, F]

### 4. Collect predictions

In [ ]:
df_test_pred  = collect_DB_preds(test_groups,  
                                 df_test,  
                                 model, 
                                 target_scaler)